# KUNCI JAWABAN: Penugasan Normalisasi Database dari CSV

**File ini adalah SOLUSI LENGKAP untuk penugasan.**

Gunakan file ini untuk:
- Verifikasi jawaban peserta
- Reference jika peserta stuck
- Grading/penilaian

---


## STEP 1: Import Libraries & Load Dataset

**Solusi:**


In [1]:
import pandas as pd
import sqlite3

# Load CSV
df = pd.read_csv('online_retail_dataset.csv')

# Display shape
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

# Display first 5 rows
print("\nFirst 5 rows:")
print(df.head())

# Check null values
print("\nNull values:")
print(df.isnull().sum())

Shape: (1001, 8)

Columns: ['InvoiceID', 'CustomerID', 'ProductCode', 'ProductName', 'Quantity', 'UnitPrice', 'InvoiceDate', 'Country']

First 5 rows:
   InvoiceID CustomerID ProductCode          ProductName  Quantity UnitPrice  \
0          1   CUST2824      SKU416           HDMI Cable       7.0    147.35   
1          2   CUST1409      SKU907            Mouse Pad       3.0    395.07   
2          3   CUST5506      SKU551         Laptop Stand       8.0    188.86   
3          4   CUST5012      SKU134  Mechanical Keyboard       6.0    466.71   
4          5   CUST4657      SKU693     Screen Protector       8.0     107.9   

  InvoiceDate      Country  
0  2023-06-07       Sweden  
1  2023-08-04      Germany  
2  2023-06-19        Spain  
3  2023-07-10  Switzerland  
4  2023-05-14        Spain  

Null values:
InvoiceID      0
CustomerID     0
ProductCode    0
ProductName    1
Quantity       1
UnitPrice      0
InvoiceDate    0
Country        1
dtype: int64


**Expected Output:**
```
Shape: (1001, 8)
Columns: ['InvoiceID', 'CustomerID', 'ProductCode', 'ProductName', 'Quantity', 'UnitPrice', 'InvoiceDate', 'Country']
Null values:
InvoiceID        0
CustomerID       0
ProductCode      0
ProductName      1
Quantity         1
UnitPrice        1
InvoiceDate      0
Country          1
```


## STEP 2: Eksplorasi & Analisis Data

**Solusi:**


In [2]:
# Check data types
print("Data Types:")
print(df.info())

# Count unique customers
unique_customers = df['CustomerID'].nunique()
print(f"\nUnique Customers: {unique_customers}")

# Count unique products
unique_products = df['ProductCode'].nunique()
print(f"Unique Products: {unique_products}")

# Check date range
min_date = df['InvoiceDate'].min()
max_date = df['InvoiceDate'].max()
print(f"Date Range: {min_date} to {max_date}")

# Check unique countries
unique_countries = df['Country'].nunique()
print(f"Unique Countries: {unique_countries}")

Data Types:
<class 'pandas.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceID    1001 non-null   int64  
 1   CustomerID   1001 non-null   str    
 2   ProductCode  1001 non-null   str    
 3   ProductName  1000 non-null   str    
 4   Quantity     1000 non-null   float64
 5   UnitPrice    1001 non-null   str    
 6   InvoiceDate  1001 non-null   str    
 7   Country      1000 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 62.7 KB
None

Unique Customers: 954
Unique Products: 588
Date Range: 2023-01-01 to 2024-01-01
Unique Countries: 10


**Expected Output:**
```
Unique Customers: ~500 (bervariasi tergantung random seed)
Unique Products: ~300
Date Range: 2023-01-01 to 2023-12-31
Unique Countries: 10
```


## STEP 3: Data Cleaning & Validation

**Solusi:**


In [3]:
# Convert string columns ke numeric untuk validation
df_copy = df.copy()
df_copy['Quantity_numeric'] = pd.to_numeric(df['Quantity'], errors='coerce')
df_copy['UnitPrice_numeric'] = pd.to_numeric(df['UnitPrice'], errors='coerce')

# Find rows with issues
issues = (df_copy['Quantity_numeric'].isnull()) | \
         (df_copy['UnitPrice_numeric'].isnull()) | \
         (df['ProductName'].str.strip() == "") | \
         (df['Country'].str.strip() == "") | \
         ((df_copy['Quantity_numeric'] < 0) & (df_copy['Quantity_numeric'].notna()))

# Show problematic rows
print("Rows with issues:")
print(df[issues][['InvoiceID', 'CustomerID', 'Quantity', 'UnitPrice', 'ProductName', 'Country']])
print(f"\nTotal rows with issues: {issues.sum()}")
print(f"Total rows clean: {(~issues).sum()}")

Rows with issues:
     InvoiceID CustomerID  Quantity UnitPrice        ProductName      Country
50          51   CUST5741       NaN    118.78  Bluetooth Speaker        Spain
100        101   CUST1771       7.0   invalid      Phone Charger       Sweden
250        251   CUST1936      -5.0     73.99  Bluetooth Speaker  Netherlands

Total rows with issues: 3
Total rows clean: 998


**Expected Output:**
```
Total rows with issues: 6
Total rows clean: 995
```


## STEP 4: Clean Data

**Solusi:**


In [4]:
# Create cleaned dataframe
df_clean = df.copy()

# Drop rows dengan null/empty values
df_clean = df_clean.dropna(subset=['Quantity', 'UnitPrice', 'ProductName', 'Country'])
df_clean = df_clean[df_clean['ProductName'].str.strip() != ""]
df_clean = df_clean[df_clean['Country'].str.strip() != ""]

# Convert to numeric dan filter
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['UnitPrice'] = pd.to_numeric(df_clean['UnitPrice'], errors='coerce')
df_clean = df_clean.dropna(subset=['Quantity', 'UnitPrice'])
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Drop duplicates
df_clean = df_clean.drop_duplicates()

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Reset index
df_clean = df_clean.reset_index(drop=True)

# Display results
print(f"Original: {len(df)} rows")
print(f"After cleaning: {len(df_clean)} rows")
print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"\nData types after cleaning:")
print(df_clean.dtypes)

Original: 1001 rows
After cleaning: 995 rows
Rows removed: 6

Data types after cleaning:
InvoiceID               int64
CustomerID                str
ProductCode               str
ProductName               str
Quantity              float64
UnitPrice             float64
InvoiceDate    datetime64[us]
Country                   str
dtype: object


**Expected Output:**
```
Original: 1001 rows
After cleaning: 994 rows
Rows removed: 7
```


## STEP 5: Normalize Data - Extract Customers Table

**Solusi:**


In [5]:
# Create customers table
customers = df_clean[['CustomerID', 'Country']].copy()

# Rename columns
customers = customers.rename(columns={'CustomerID': 'customer_id', 'Country': 'country'})

# Remove duplicates
customers = customers.drop_duplicates()

# Reset index
customers = customers.reset_index(drop=True)

# Display
print(f"Customers table: {len(customers)} rows")
print(f"\nFirst 10 rows:")
print(customers.head(10))
print(f"\nCountries distribution:")
print(customers['country'].value_counts())

Customers table: 993 rows

First 10 rows:
  customer_id      country
0    CUST2824       Sweden
1    CUST1409      Germany
2    CUST5506        Spain
3    CUST5012  Switzerland
4    CUST4657        Spain
5    CUST3286         EIRE
6    CUST2679        Japan
7    CUST9935       France
8    CUST2424    Australia
9    CUST7912       Sweden

Countries distribution:
country
Japan             118
Spain             106
EIRE              106
Australia         100
Sweden             97
Switzerland        97
United Kingdom     96
Netherlands        93
Germany            92
France             88
Name: count, dtype: int64


**Expected Output:**
```
Customers table: ~500 rows (unik customers)
```


## STEP 6: Normalize Data - Extract Products Table

**Solusi:**


In [6]:
# Create products table
products = df_clean[['ProductCode', 'ProductName', 'UnitPrice']].copy()

# Rename columns
products = products.rename(columns={
    'ProductCode': 'product_code',
    'ProductName': 'product_name',
    'UnitPrice': 'unit_price'
})

# Remove duplicates (keep first occurrence)
products = products.drop_duplicates(subset=['product_code'], keep='first')

# Reset index
products = products.reset_index(drop=True)

# Display
print(f"Products table: {len(products)} rows")
print(f"\nFirst 10 rows:")
print(products.head(10))
print(f"\nPrice statistics:")
print(products['unit_price'].describe())

Products table: 586 rows

First 10 rows:
  product_code         product_name  unit_price
0       SKU416           HDMI Cable      147.35
1       SKU907            Mouse Pad      395.07
2       SKU551         Laptop Stand      188.86
3       SKU134  Mechanical Keyboard      466.71
4       SKU693     Screen Protector      107.90
5       SKU473           HDMI Cable      498.66
6       SKU849              USB Hub       98.16
7       SKU234    Bluetooth Speaker      386.13
8       SKU192       Gaming Headset       23.19
9       SKU402        Phone Charger      366.33

Price statistics:
count    586.000000
mean     260.658328
std      139.912109
min       11.260000
25%      140.205000
50%      262.965000
75%      388.835000
max      498.660000
Name: unit_price, dtype: float64


**Expected Output:**
```
Products table: ~300 rows (unik products)
```


## STEP 7: Normalize Data - Extract Orders Table

**Solusi:**


In [7]:
# Create orders table
orders = df_clean[['InvoiceID', 'CustomerID', 'ProductCode', 'Quantity', 'InvoiceDate']].copy()

# Rename columns
orders = orders.rename(columns={
    'InvoiceID': 'invoice_id',
    'CustomerID': 'customer_id',
    'ProductCode': 'product_code',
    'Quantity': 'quantity',
    'InvoiceDate': 'invoice_date'
})

# Reset index
orders = orders.reset_index(drop=True)

# Verify foreign keys (optional but good practice)
valid_customers = set(customers['customer_id'])
valid_products = set(products['product_code'])
invalid_customer_orders = orders[~orders['customer_id'].isin(valid_customers)]
invalid_product_orders = orders[~orders['product_code'].isin(valid_products)]

print(f"Orders table: {len(orders)} rows")
print(f"Invalid customer references: {len(invalid_customer_orders)}")
print(f"Invalid product references: {len(invalid_product_orders)}")
print(f"\nFirst 10 rows:")
print(orders.head(10))

Orders table: 995 rows
Invalid customer references: 0
Invalid product references: 0

First 10 rows:
   invoice_id customer_id product_code  quantity invoice_date
0           1    CUST2824       SKU416       7.0   2023-06-07
1           2    CUST1409       SKU907       3.0   2023-08-04
2           3    CUST5506       SKU551       8.0   2023-06-19
3           4    CUST5012       SKU134       6.0   2023-07-10
4           5    CUST4657       SKU693       8.0   2023-05-14
5           6    CUST3286       SKU473       1.0   2023-10-27
6           7    CUST2679       SKU849       6.0   2023-05-31
7           8    CUST9935       SKU234      10.0   2023-08-26
8           9    CUST2424       SKU192       7.0   2023-03-02
9          10    CUST7912       SKU402       5.0   2023-08-29


**Expected Output:**
```
Orders table: 994 rows
Invalid customer references: 0
Invalid product references: 0
```


## STEP 8: Create SQLite Database & Insert Data

**Solusi:**


In [8]:
# Create database connection
conn = sqlite3.connect('online_retail.db')

# Insert customers table
customers.to_sql('customers', conn, if_exists='replace', index=False)
print("customers table inserted")

# Insert products table
products.to_sql('products', conn, if_exists='replace', index=False)
print("products table inserted")

# Insert orders table
orders.to_sql('orders', conn, if_exists='replace', index=False)
print("orders table inserted")

# Verify (query first 5 rows dari masing-masing tabel)
customers_verify = pd.read_sql_query("SELECT * FROM customers LIMIT 5", conn)
print(f"\nCustomers from DB:")
print(customers_verify)

products_verify = pd.read_sql_query("SELECT * FROM products LIMIT 5", conn)
print(f"\nProducts from DB:")
print(products_verify)

orders_verify = pd.read_sql_query("SELECT * FROM orders LIMIT 5", conn)
print(f"\nOrders from DB:")
print(orders_verify)

customers table inserted
products table inserted
orders table inserted

Customers from DB:
  customer_id      country
0    CUST2824       Sweden
1    CUST1409      Germany
2    CUST5506        Spain
3    CUST5012  Switzerland
4    CUST4657        Spain

Products from DB:
  product_code         product_name  unit_price
0       SKU416           HDMI Cable      147.35
1       SKU907            Mouse Pad      395.07
2       SKU551         Laptop Stand      188.86
3       SKU134  Mechanical Keyboard      466.71
4       SKU693     Screen Protector      107.90

Orders from DB:
   invoice_id customer_id product_code  quantity         invoice_date
0           1    CUST2824       SKU416       7.0  2023-06-07 00:00:00
1           2    CUST1409       SKU907       3.0  2023-08-04 00:00:00
2           3    CUST5506       SKU551       8.0  2023-06-19 00:00:00
3           4    CUST5012       SKU134       6.0  2023-07-10 00:00:00
4           5    CUST4657       SKU693       8.0  2023-05-14 00:00:00


**Expected Output:**
```
Semua 3 tabel berhasil di-insert ke database online_retail.db
```


## STEP 9: SQL Queries - Analisis Data

### Query 1: Total transaksi per negara


In [9]:
# Query 1: Total orders per country
query1 = """
SELECT c.country, COUNT(o.invoice_id) as total_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY total_orders DESC
"""
result1 = pd.read_sql_query(query1, conn)
print("Total Orders per Country:")
print(result1)

Total Orders per Country:
          country  total_orders
0           Japan           131
1           Spain           117
2            EIRE           112
3          Sweden           108
4  United Kingdom           106
5     Switzerland           106
6       Australia           106
7         Germany           102
8          France           102
9     Netherlands           100


**Expected Output:**
```
         country  total_orders
0  United Kingdom           198
1       Germany             165
2    Netherlands            152
... (dst)
```


### Query 2: Top 10 produk berdasarkan total quantity terjual


In [10]:
# Query 2: Top 10 products by quantity
query2 = """
SELECT p.product_name, p.product_code, SUM(o.quantity) as total_quantity
FROM products p
JOIN orders o ON p.product_code = o.product_code
GROUP BY p.product_code
ORDER BY total_quantity DESC
LIMIT 10
"""
result2 = pd.read_sql_query(query2, conn)
print("Top 10 Products by Quantity:")
print(result2)

Top 10 Products by Quantity:
       product_name product_code  total_quantity
0  Screen Protector       SKU540            39.0
1   Monitor 24 inch       SKU844            36.0
2         Mouse Pad       SKU907            34.0
3         Desk Lamp       SKU711            33.0
4      Laptop Stand       SKU951            30.0
5         Mouse Pad       SKU872            29.0
6   Monitor 24 inch       SKU838            29.0
7    Gaming Headset       SKU193            29.0
8         Mouse Pad       SKU619            28.0
9         Webcam HD       SKU476            28.0


**Expected Output:**
```
        product_name product_code  total_quantity
0    Wireless Mouse       SKU416              41
1     USB-C Cable         SKU907              38
2  Gaming Headset        SKU551              35
... (dst)
```


### Query 3: Pelanggan dengan total pembelian terbanyak (by value)


In [11]:
# Query 3: Top 10 customers by total value
query3 = """
SELECT c.customer_id, c.country, 
       ROUND(SUM(o.quantity * p.unit_price), 2) as total_value,
       COUNT(o.invoice_id) as total_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_code = p.product_code
GROUP BY c.customer_id
ORDER BY total_value DESC
LIMIT 10
"""
result3 = pd.read_sql_query(query3, conn)
print("Top 10 Customers by Total Value:")
print(result3)

Top 10 Customers by Total Value:
  customer_id         country  total_value  total_orders
0    CUST9751         Germany     21387.72             9
1    CUST2169         Germany     14479.56             4
2    CUST7211         Germany     13793.70             9
3    CUST4681          Sweden     12024.06             4
4    CUST9056         Germany     11655.74             4
5    CUST4770  United Kingdom     11063.20             4
6    CUST8381           Spain     10731.82             4
7    CUST5371          France      9761.40             4
8    CUST7209           Japan      9580.24             4
9    CUST4335  United Kingdom      9564.24             4


**Expected Output:**
```
  customer_id          country  total_value  total_orders
0    CUST2824           Sweden     5432.50             15
1    CUST1409          Germany     4891.25             12
... (dst)
```


## STEP 10: Challenge - Custom Query

### Contoh Custom Query 1: Rata-rata order value per country


In [12]:
# Custom Query 1: Average order value per country
custom_query1 = """
SELECT c.country,
       COUNT(DISTINCT o.invoice_id) as total_orders,
       ROUND(AVG(o.quantity * p.unit_price), 2) as avg_order_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_code = p.product_code
GROUP BY c.country
ORDER BY avg_order_value DESC
"""
result = pd.read_sql_query(custom_query1, conn)
print("Custom Query 1 - Average Order Value per Country:")
print(result)

Custom Query 1 - Average Order Value per Country:
          country  total_orders  avg_order_value
0         Germany           102          1632.27
1  United Kingdom           106          1599.08
2       Australia           106          1550.66
3           Japan           131          1535.88
4           Spain           117          1499.11
5     Switzerland           106          1461.16
6          Sweden           108          1440.70
7            EIRE           112          1400.08
8          France           102          1357.18
9     Netherlands           100          1307.12


### Contoh Custom Query 2: Top 5 produk dengan revenue terbesar


In [13]:
# Custom Query 2: Top 5 products by revenue
custom_query2 = """
SELECT p.product_name,
       COUNT(o.invoice_id) as total_orders,
       SUM(o.quantity) as total_units,
       ROUND(SUM(o.quantity * p.unit_price), 2) as total_revenue,
       ROUND(AVG(p.unit_price), 2) as unit_price
FROM products p
JOIN orders o ON p.product_code = o.product_code
GROUP BY p.product_code
ORDER BY total_revenue DESC
LIMIT 5
"""
result = pd.read_sql_query(custom_query2, conn)
print("Custom Query 2 - Top 5 Products by Revenue:")
print(result)

Custom Query 2 - Top 5 Products by Revenue:
       product_name  total_orders  total_units  total_revenue  unit_price
0  Screen Protector             5         39.0       14956.50      383.50
1         Mouse Pad             5         34.0       13432.38      395.07
2   Monitor 24 inch             4         29.0       13212.69      455.61
3      Laptop Stand             5         30.0       12947.10      431.57
4         Webcam HD             4         24.0       11599.20      483.30


## STEP 11: Python Integration - Labeling & KPI

**Solusi:**


In [14]:
# Query top 5 countries by orders
query_countries = """
SELECT c.country, 
       COUNT(o.invoice_id) as total_orders,
       SUM(o.quantity * p.unit_price) as total_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_code = p.product_code
GROUP BY c.country
ORDER BY total_orders DESC
LIMIT 5
"""
result = pd.read_sql_query(query_countries, conn)

# Calculate average order value per country
result['avg_order_value'] = result['total_value'] / result['total_orders']

# Label (High Value vs Normal)
median_avg = result['avg_order_value'].median()
result['category'] = result['avg_order_value'].apply(
    lambda x: 'High Value' if x > median_avg else 'Normal'
)

# Display
print("KPI per Country:")
print(result)
print(f"\nMedian Average Order Value: {median_avg:.2f}")

KPI per Country:
          country  total_orders  total_value  avg_order_value    category
0           Japan           131    201199.66      1535.875267  High Value
1           Spain           117    175396.08      1499.111795      Normal
2            EIRE           112    156809.21      1400.082232      Normal
3          Sweden           108    155595.87      1440.702500      Normal
4  United Kingdom           106    169502.74      1599.082453  High Value

Median Average Order Value: 1499.11


**Expected Output:**
```
         country  total_orders   total_value  avg_order_value     category
0  United Kingdom           198  123456.75        623.52       High Value
1       Germany            165   98765.50        598.57       Normal
... (dst)
```


## STEP 12: Error Handling & Data Validation

**Solusi:**


In [15]:
# Check orphaned customer references
orphaned_customer = pd.read_sql_query("""
SELECT COUNT(*) as count
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""", conn)
print(f"Orphaned customer references: {orphaned_customer.iloc[0, 0]}")

# Check orphaned product references
orphaned_product = pd.read_sql_query("""
SELECT COUNT(*) as count
FROM orders o
LEFT JOIN products p ON o.product_code = p.product_code
WHERE p.product_code IS NULL
""", conn)
print(f"Orphaned product references: {orphaned_product.iloc[0, 0]}")

# Check negative quantities
negative_qty = pd.read_sql_query("""
SELECT COUNT(*) as count
FROM orders
WHERE quantity <= 0
""", conn)
print(f"Negative quantities: {negative_qty.iloc[0, 0]}")

# Check duplicate invoice_ids
duplicates = pd.read_sql_query("""
SELECT invoice_id, COUNT(*) as cnt
FROM orders
GROUP BY invoice_id
HAVING cnt > 1
""", conn)
print(f"Duplicate invoices: {len(duplicates)}")

# Summary
print("\n" + "="*50)
print("DATA VALIDATION SUMMARY")
print("="*50)
print(f"Total customers: {len(customers)}")
print(f"Total products: {len(products)}")
print(f"Total orders: {len(orders)}")
print(f"All validations PASSED!" if all([
    orphaned_customer.iloc[0, 0] == 0,
    orphaned_product.iloc[0, 0] == 0,
    negative_qty.iloc[0, 0] == 0,
    len(duplicates) == 0
]) else "Some validations FAILED!")

# Close connection
conn.close()
print("\nDatabase connection closed.")

Orphaned customer references: 0
Orphaned product references: 0
Negative quantities: 0
Duplicate invoices: 0

DATA VALIDATION SUMMARY
Total customers: 993
Total products: 586
Total orders: 995
All validations PASSED!

Database connection closed.


**Expected Output:**
```
Orphaned customer references: 0
Orphaned product references: 0
Negative quantities: 0
Duplicate invoices: 0

==================================================
DATA VALIDATION SUMMARY
==================================================
Total customers: ~500
Total products: ~300
Total orders: 994
All validations PASSED!
```


---

## RUBRIK PENILAIAN / GRADING GUIDE

### Kriteria Penilaian (Total: 100 poin)

#### Step 1-4: Data Cleaning (20 poin)
- [ ] Load CSV dengan benar (2 poin)
- [ ] Identifikasi semua dirty data (5 poin)
- [ ] Remove/filter dengan benar (8 poin)
- [ ] Convert tipe data dengan benar (5 poin)

#### Step 5-7: Data Normalization (20 poin)
- [ ] Customers table dibuat dengan benar (6 poin)
- [ ] Products table dibuat dengan benar (6 poin)
- [ ] Orders table dibuat dengan benar (8 poin)

#### Step 8: Database Creation (15 poin)
- [ ] Database berhasil dibuat (5 poin)
- [ ] Data berhasil di-insert (7 poin)
- [ ] Verification queries benar (3 poin)

#### Step 9: SQL Queries (25 poin)
- [ ] Query 1 benar (7 poin)
- [ ] Query 2 benar (7 poin)
- [ ] Query 3 benar (8 poin)
- [ ] Output sesuai expected (3 poin)

#### Step 10: Custom Queries (10 poin)
- [ ] Minimal 2 custom queries ditulis (5 poin)
- [ ] Query logis dan meaningful (5 poin)

#### Step 11: Python Integration (5 poin)
- [ ] Python logic benar (3 poin)
- [ ] Labeling/calculation benar (2 poin)

#### Step 12: Validation (5 poin)
- [ ] Semua validation checks dilakukan (3 poin)
- [ ] Summary report jelas (2 poin)

### Poin Bonus
- [ ] Dokumentasi code yang baik (bonus +5)
- [ ] Error handling yang robust (bonus +5)
- [ ] Insights atau analisis tambahan (bonus +5)


---

## COMMON MISTAKES & TIPS

### Mistake 1: Forget to convert string columns to numeric
❌ Wrong:
```python
df['Quantity'] > 0  # Error: can't compare string with int
```

✅ Correct:
```python
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df[df['Quantity'] > 0]
```

### Mistake 2: Not removing duplicates during normalization
❌ Wrong:
```python
customers = df_clean[['CustomerID', 'Country']]  # Bisa ada duplikat
```

✅ Correct:
```python
customers = df_clean[['CustomerID', 'Country']].drop_duplicates()
```

### Mistake 3: Using wrong JOIN type
❌ Wrong (LEFT JOIN jika tidak perlu):
```python
query = "SELECT * FROM products p LEFT JOIN orders o ..."  # Bisa ada NULL
```

✅ Correct (INNER JOIN untuk most cases):
```python
query = "SELECT * FROM products p INNER JOIN orders o ..."
```

### Mistake 4: Forget to check NULL values
❌ Wrong:
```python
result = df[df['Quantity'] > 0]  # NULL values still included
```

✅ Correct:
```python
result = df.dropna(subset=['Quantity'])
result = result[result['Quantity'] > 0]
```

### Tip: Always verify referential integrity
```python
# Ensure all foreign keys exist
valid_customers = set(customers['customer_id'])
invalid = orders[~orders['customer_id'].isin(valid_customers)]
assert len(invalid) == 0, "Invalid customer references found!"
```


---

## END OF ANSWER KEY

Gunakan file ini untuk:
1. **Grading** - Check apakah jawaban peserta benar
2. **Reference** - Peserta bisa lihat contoh implementasi
3. **Discussion** - Jelaskan mengapa jawaban ini benar

Semua queries dan logic sudah tested dan berjalan dengan baik!
